### Q1

In [ ]:
import re
from pathlib import Path
import pandas as pd

roll_number = "1024170160"
category_names = ["billing", "account", "general"]
last_two_digits = [int(digit) for digit in roll_number[-2:]]
personalized_categories = [category_names[digit % 3] for digit in last_two_digits]

fixed_entries = [
    {"question": "what is the annual fee", "answer": "The annual fee is Rs 500.", "keywords": "fee cost price charge", "category": "billing"},
    {"question": "how to reset password", "answer": "Go to Settings > Reset Password.", "keywords": "password reset login", "category": "account"},
    {"question": "what are your working hours", "answer": "We are open 9 AM to 5 PM.", "keywords": "hours timing open time", "category": "general"},
    {"question": "how can i pay the fee", "answer": "You can pay via UPI, card, or net banking.", "keywords": "pay payment upi fee", "category": "billing"}
]
personalized_entries = [
    {"question": "how can i download a payment receipt", "answer": "Open the payments section of your student portal and choose the transaction to download its receipt.", "keywords": "invoice receipt transaction", "category": personalized_categories[0]},
    {"question": "how can i review the charges on my student account", "answer": "Open your student account statement or contact the billing office for an itemized list of charges.", "keywords": "statement balance charges", "category": personalized_categories[1]}
]
faq_data = pd.DataFrame(fixed_entries + personalized_entries, columns=["question", "answer", "keywords", "category"])
print("Roll number:", roll_number)
print("Personalized categories for final digits:", list(zip(last_two_digits, personalized_categories)))
print(faq_data)
assert len(faq_data) == 6

### Q2

In [ ]:
def query_tokens(text):
    return set(re.findall(r"[a-z0-9]+", text.lower()))

def score_query(query, df):
    tokens = query_tokens(query)
    if not tokens:
        return pd.DataFrame(columns=["question", "answer", "keywords", "category", "confidence"])
    results = []
    for _, entry in df.iterrows():
        entry_tokens = query_tokens(f"{entry['question']} {entry['keywords']}")
        confidence = len(tokens & entry_tokens) / len(tokens)
        if confidence > 0:
            results.append({**entry.to_dict(), "confidence": confidence})
    return pd.DataFrame(results).sort_values(
        ["confidence", "question"], ascending=[False, True], ignore_index=True
    )

ranked_matches = score_query("annual fee", faq_data)
print(ranked_matches.to_string(index=False))

### Q3

In [ ]:
def same_category(category_name, df):
    return df.loc[df["category"].str.lower() == category_name.lower(), ["question", "answer", "keywords", "category"]]

personalized_category = personalized_entries[0]["category"]
print(f"Personalized category: {personalized_category}")
print(same_category(personalized_category, faq_data).to_string(index=False))

### Q4

In [ ]:
entry_index = 0
new_keyword = input(f"Enter one new keyword for FAQ entry '{faq_data.loc[entry_index, 'question']}': ").strip().lower()
if not re.fullmatch(r"[a-z0-9_-]+", new_keyword):
    raise ValueError("Enter one non-empty keyword using letters, numbers, underscores, or hyphens.")
existing_keywords = faq_data.loc[entry_index, "keywords"].split()
if new_keyword not in existing_keywords:
    faq_data.loc[entry_index, "keywords"] = " ".join(existing_keywords + [new_keyword])

output_dir = Path.cwd()
if output_dir.name != "A4" and (output_dir / "A4").is_dir():
    output_dir = output_dir / "A4"
csv_path = output_dir / "1024170160_faq_data.csv"
faq_data.to_csv(csv_path, index=False)
print("Updated keywords:", faq_data.loc[entry_index, "keywords"])
print("Saved:", csv_path.resolve())

### Q5

In [ ]:
faq_counts = faq_data.groupby("category").size().sort_index()
print("FAQ entries per category:")
print(faq_counts)

### Q6

In [ ]:
def answer_query(query, df):
    matches = score_query(query, df)
    if matches.empty:
        print("No matching FAQ entry found.")
        return matches
    highest_score = matches["confidence"].iloc[0]
    best_matches = matches[matches["confidence"] == highest_score]
    if len(best_matches) > 1:
        print(f"Tie: {len(best_matches)} entries share the highest confidence ({highest_score:.2f}).")
        print(best_matches.to_string(index=False))
    else:
        print(f"Best match (confidence {highest_score:.2f}):")
        print(best_matches.to_string(index=False))
    return best_matches

print("Tie demonstration:")
tie_matches = answer_query("fee", faq_data)
print("\nNon-tie demonstration:")
single_match = answer_query("reset password", faq_data)
assert len(tie_matches) == 2
assert len(single_match) == 1